# 02c | Initialize from seed weights for a new expanded-corpus stage

130U-FinLLM | 130U Financial Language Model

Complete 01b and verify all three backups. This notebook requires a real seed_v1 pretraining checkpoint with at least 300 updates,
freezes its identity and retains the original 8K tokenizer. Because the corpus changes, initialize from weights while creating new optimizer and scheduler states,
data cursors and stage counts. Preserve the original seed checkpoint; this is not an exact resume of the same data stream.

By default verify update 0 to 1, then train to 610 cumulative updates in the new stage, nominally about 10M new token positions.
Reports provide actual effective tokens; the parent model's approximately 4.91M tokens are listed separately. The learning-rate schedule spans 6,104 updates,
so reaching 610 updates does not complete the 100M-token budget or the full financial model.

Select Runtime > Change runtime type > T4 GPU, then run cells in order.
Rerun the same version with strict recovery. After interruption, reconnect and confirm training has stopped before rerunning from the first cell.
Do not start a second training process concurrently. Use a separate experiment version when changing data, tokenizer or execution semantics.

In [ ]:
import os, sys, json, shutil, subprocess, importlib
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # Resolve and display the commit first; use the full 40-character SHA for reproduction.
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF does not resolve uniquely; enter the full commit SHA.")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM belongs to another repository; choose a new directory.")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("Pinned code revision:", REVISION)

# Keep the bundled Colab CUDA / PyTorch; install only the other project dependencies.
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} is incompatible with the existing environment:{incompatible}")
    print("tokenizers compatibility check passed:", installed)

def check_project_dependencies():
    # Read the refreshed installed metadata, not the previous editable checkout.
    problems = []
    for value in metadata.requires("finllm-130u") or []:
        requirement = Requirement(value)
        if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            problems.append(f"Missing {requirement.name}")
            continue
        if installed not in requirement.specifier:
            problems.append(f"{requirement.name}=={installed} does not satisfy {requirement}")
    if problems:
        raise RuntimeError("Project dependency check failed:" + "; ".join(problems))
    print("Installed project dependencies passed validation.")

def activate_project_in_kernel():
    # pip installs the editable .pth for NEW Python processes. A notebook kernel
    # that was already running does not process that .pth again automatically.
    source = (PROJECT / "src").resolve()
    expected = source / "finllm"
    if not (expected / "__init__.py").is_file():
        raise RuntimeError(f"Project Python source is missing:{expected}")
    sys.path[:] = [str(source)] + [entry for entry in sys.path if entry != str(source)]
    # Git may have changed code in the same directory since an earlier cell run.
    # Refresh only our modules; never reload Torch or its live CUDA state.
    for name in tuple(sys.modules):
        if name == "finllm" or name.startswith("finllm."):
            del sys.modules[name]
    importlib.invalidate_caches()
    package = importlib.import_module("finllm")
    runtime = importlib.import_module("finllm.runtime")
    for module, filename in ((package, "__init__.py"), (runtime, "runtime.py")):
        if Path(module.__file__).resolve() != expected / filename:
            raise RuntimeError(f"The notebook imported the wrong project location:{module.__file__}")
    if not callable(runtime.restore_seed_checkpoint):
        raise RuntimeError("The checkpoint restore interface is unavailable in this notebook.")
    print("Current notebook kernel import check passed:", runtime.__file__)

def check_fresh_process_import():
    # Verify the installed editable package separately, without injecting src
    # into PYTHONPATH. Both the current kernel and future CLI processes must work.
    probe = """import json, sys
from pathlib import Path
import finllm
import finllm.runtime as runtime
expected = Path(sys.argv[1]).resolve() / 'finllm'
for module, filename in ((finllm, '__init__.py'), (runtime, 'runtime.py')):
    if Path(module.__file__).resolve() != expected / filename:
        raise RuntimeError('Fresh process imported the wrong project: ' + str(module.__file__))
if not callable(runtime.restore_seed_checkpoint):
    raise RuntimeError('Fresh process checkpoint restore is unavailable')
print(json.dumps({'finllm': str(Path(finllm.__file__).resolve()), 'runtime': str(Path(runtime.__file__).resolve())}))
"""
    result = subprocess.check_output([sys.executable, "-c", probe, str(PROJECT / "src")],
                                     cwd=PROJECT, text=True)
    print("Fresh Python process import check passed:", result.strip())

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("PyTorch >= 2.5 is required; select a newer Colab runtime.")
tokenizer_requirements = tokenizer_install_requirements()
print("Active tokenizers constraints:", tokenizer_requirements)
torch_version_before = metadata.version("torch")
torch_file_before = Path(torch.__file__).resolve()
# One resolver transaction replaces the previous finllm metadata AND its
# dependencies. Upgrading tokenizers first leaves old finllm bounds active and
# produces a misleading conflict warning even when the eventual install works.
# The exact installed Torch build is constrained, so pip keeps Colab's CUDA build.
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".",
        "torch==" + torch_version_before, *tokenizer_requirements)
importlib.invalidate_caches()
if metadata.version("torch") != torch_version_before or Path(torch.__file__).resolve() != torch_file_before:
    raise RuntimeError("The PyTorch environment changed unexpectedly; preserve the output and stop training.")
check_tokenizer_compatibility(tokenizer_requirements)
check_project_dependencies()
activate_project_in_kernel()
check_fresh_process_import()
print("Notebook setup complete; PyTorch remains:", torch_version_before)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Stage inputs are missing; prepare them or restore them from Drive first:\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"Free disk space in /content:{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
MOUNT_DRIVE = True
if not MOUNT_DRIVE:
    raise RuntimeError("This stage requires existing seed inputs and versioned backups; mount Drive first.")
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive project directory:", DRIVE_ROOT)

In [ ]:
EXPANSION_RETENTION_BEFORE_SUCCEEDED = False
EXPANSION_PROBE_SUCCEEDED = False
EXPANSION_TRAIN_SUCCEEDED = False
EXPANSION_RETENTION_AFTER_SUCCEEDED = False
EXPANSION_TRAIN_BACKED_UP = False
SEED_VERSION = "seed_v1"
EXPANSION_VERSION = "expansion_v1"
MIN_PARENT_UPDATES = 300
PROBE_UPDATES = 1
TARGET_UPDATES = 610
if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU runtime; the expansion stage does not start on CPU.")
from finllm.expansion_runtime import prepare_expansion_training, run_expansion_segment, retention_check, backup_expansion_training
expansion = prepare_expansion_training(
    PROJECT, DRIVE_ROOT, EXPANSION_VERSION, SEED_VERSION,
    min_parent_updates=MIN_PARENT_UPDATES, probe_updates=PROBE_UPDATES,
    target_updates=TARGET_UPDATES)
print("Restored new-stage update count:", expansion["restored_updates"])
print("Frozen parent checkpoint identity:", json.dumps(expansion["parent"], indent=2, ensure_ascii=False))
print("New-stage configuration:", json.dumps(expansion["config"], indent=2, ensure_ascii=False))
print("GPU:", torch.cuda.get_device_name(0))

Measure bounded loss for the frozen parent checkpoint on original seed general-validation shards, then compare identical positions after training.
Also retain validation loss for the expanded corpus. Both diagnose text prediction, not financial stance.
Read existing measurements when all input identities match.

In [ ]:
EXPANSION_RETENTION_BEFORE_SUCCEEDED = False
EXPANSION_PROBE_SUCCEEDED = False
EXPANSION_TRAIN_SUCCEEDED = False
EXPANSION_RETENTION_AFTER_SUCCEEDED = False
EXPANSION_TRAIN_BACKED_UP = False
retention_before = retention_check(expansion, "before")
print(json.dumps(retention_before, indent=2, ensure_ascii=False))
EXPANSION_RETENTION_BEFORE_SUCCEEDED = True

In [ ]:
EXPANSION_PROBE_SUCCEEDED = False
EXPANSION_TRAIN_SUCCEEDED = False
EXPANSION_RETENTION_AFTER_SUCCEEDED = False
EXPANSION_TRAIN_BACKED_UP = False
if not globals().get("EXPANSION_RETENTION_BEFORE_SUCCEEDED", False):
    raise RuntimeError("Complete the parent checkpoint measurement on the original validation set first.")
expansion_probe = run_expansion_segment(expansion, "probe", finllm)
print(json.dumps(expansion_probe, indent=2, ensure_ascii=False))
EXPANSION_PROBE_SUCCEEDED = True

Train to 610 new-stage updates only after the 0-to-1 check succeeds. Validate and restore later checkpoints of the same version without rolling back.
Save and sync to Drive every 10 updates. Reported updates/tokens are cumulative within the new stage; elapsed time covers only that call.
Incompatible data/tokenizer/configuration/runtime stops execution; do not disable provenance checks or use random initialization to bypass errors.

In [ ]:
EXPANSION_TRAIN_SUCCEEDED = False
EXPANSION_RETENTION_AFTER_SUCCEEDED = False
EXPANSION_TRAIN_BACKED_UP = False
if not globals().get("EXPANSION_RETENTION_BEFORE_SUCCEEDED", False) or not globals().get("EXPANSION_PROBE_SUCCEEDED", False):
    raise RuntimeError("Complete the original-validation measurement and one-step weight-handoff check first.")
expansion_result = run_expansion_segment(expansion, "continue", finllm)
print(json.dumps(expansion_result, indent=2, ensure_ascii=False))
EXPANSION_TRAIN_SUCCEEDED = True

In [ ]:
EXPANSION_RETENTION_AFTER_SUCCEEDED = False
EXPANSION_TRAIN_BACKED_UP = False
if not globals().get("EXPANSION_TRAIN_SUCCEEDED", False):
    raise RuntimeError("New-stage training has not completed successfully; keep existing backups.")
retention_after = retention_check(expansion, "after")
print(json.dumps(retention_after, indent=2, ensure_ascii=False))
EXPANSION_RETENTION_AFTER_SUCCEEDED = True

In [ ]:
EXPANSION_TRAIN_BACKED_UP = False
if not all(globals().get(flag, False) for flag in (
    "EXPANSION_RETENTION_BEFORE_SUCCEEDED", "EXPANSION_PROBE_SUCCEEDED",
    "EXPANSION_TRAIN_SUCCEEDED", "EXPANSION_RETENTION_AFTER_SUCCEEDED")):
    raise RuntimeError("Measurement, one-step verification and training have not all succeeded; do not report a complete stage backup.")
expansion_train_backup = backup_expansion_training(expansion)
print(json.dumps(expansion_train_backup, indent=2, ensure_ascii=False))
shutil.copy2(PROJECT / "runtime_revision.txt", expansion["paths"]["drive"] / "runtime_revision.txt")
EXPANSION_TRAIN_BACKED_UP = True
print("Expanded general training and verified backup complete:", expansion["paths"]["drive"])
print("Original seed data and 300-update artifacts preserved:", expansion["paths"]["seed_drive"])

Keep one-step and training-segment reports, retention before/after, actual token counts and the final `verified: true` backup output.
Model architecture and vocabulary are unchanged; record original seed 300 updates separately from new-stage updates.
This notebook trains only on expanded general text; it does not automatically start financial-domain training, SFT or DPO.
Notebook 01b prepares expanded financial shards. The next stage requires matching mixture configuration and entry points; human asset/stance labels require separate confirmation.